# RP-010 — paired generic versus rich supervision

Read `docs/RP010_CONTROLLED_TRAINING_AUDIT.md` and `docs/RP010_RUNBOOK.md` first.
A = class + two fixed RP-009 templates; B = class + saved detailed/generalized.
Class-name counting on standard FSC147 test is primary. FSC147-S is secondary
and shares images with training. No test-driven checkpoint selection.

**Asset bootstrap restored from the working `train_colab.ipynb` workflow:**
`T2ICount-assets.zip` on MyDrive is copied/extracted to Colab SSD
(`/content/T2ICount-assets`) for SD/CLIP and optionally FSC147. The RP-010
initialization, checkpoints, logs and outputs are stored persistently under
`/content/drive/MyDrive/T2ICount-assets/` via dedicated `rp010` directory symlinks.
The existing GitHub trainer/configs and the A/B scientific controls are unchanged.

Source is fetched from GitHub `rich-prompt-alignment` after researcher review,
commit and push. No source ZIP. This notebook does not commit, push, start full
training by default, or disconnect the runtime. Prefer an L4 runtime; its peak
memory requirement is not yet measured.

Run all cells from top to bottom. First prepare once, then smoke A and B in
separate sessions, review both reports, then train each arm separately. Resume
replays an interrupted epoch from the last complete boundary. Keep source,
packages, GPU type and immutable asset locations identical within a pair.


In [13]:
from pathlib import Path
import os, sys, subprocess, json, re, shutil, zipfile

# Mirrors the working notebooks/train_colab.ipynb asset bootstrap.
# Source pretrained weights are LOCAL on the Colab SSD, not in MyDrive/pretrained.
ASSET_ROOT = Path('/content/T2ICount-assets')
DRIVE_ASSET_ROOT = Path('/content/drive/MyDrive/T2ICount-assets')
ASSET_ZIP = Path('/content/drive/MyDrive/T2ICount-assets.zip')
LOCAL_ZIP = Path('/content/T2ICount-assets.zip')
DATA_ROOT = None  # optional explicit FSC147 folder; auto-detect extracted local or Drive data
TRAIN_BANK = DRIVE_ASSET_ROOT / 'prompts/fsc147_train1000_seed3407_gemma4_v3.json'
S_BANK = DRIVE_ASSET_ROOT / 'prompts/fsc147s_gemma4_26b_v3.json'
REPO = Path('/content/T2ICount-RP010')
REPO_URL = 'https://github.com/bqa100507-spec/T2ICount-Implementation.git'
SOURCE_BRANCH = 'rich-prompt-alignment'
EXPECTED_COMMIT = ''  # recommend pinning the reviewed full SHA for prepare + smoke + train
SOURCE_REVISION = None

STAGE = 'compare'  # prepare / smoke / train / evaluate / compare
ARM = 'A'          # switch A -> B; preparation is shared
SEED = 3407       # new seeds create new shared initialization
PAIR_ID = 'pair_v1'  # use a new ID for a new pair/smoke attempt; never overwrite
RUN_STAGE = True
ENABLE_FULL_TRAINING = False
RESUME = False
INSTALL_DEPENDENCIES = True

assert STAGE in ('prepare','smoke','train','evaluate','compare') and ARM in ('A','B')
assert re.fullmatch(r'[A-Za-z0-9_-]+', PAIR_ID)
assert not RESUME or STAGE == 'train'

def stream(command, cwd=None):
    print('+', ' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(list(map(str, command)), cwd=cwd, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait():
        raise RuntimeError(f'Command failed with exit {process.returncode}; read the first error above')

def git(*args):
    return subprocess.check_output(['git','-C',str(REPO),*args],text=True).strip()


## Runtime and Drive
An available GPU alone is not a smoke pass. Preparation checks host RAM;
smoke checks a model-derived memory budget estimate before allocating GPU state.
The estimate excludes activations and optimizer workspace. Do not alter precision,
crop size, freezing or losses to fit this comparison onto a smaller GPU.


In [14]:
if RUN_STAGE:
    import torch
    print('Python:', sys.version)
    print('Torch/CUDA:', torch.__version__, torch.version.cuda)
    assert torch.cuda.is_available(), 'Choose a Colab GPU runtime; prefer L4'
    assert torch.cuda.device_count() == 1, 'RP-010 uses exactly one GPU'
    print('GPU:', torch.cuda.get_device_name(0))
    print('Free/total GiB:', [round(x/2**30, 2) for x in torch.cuda.mem_get_info()])
    if not Path('/content/drive/MyDrive').is_dir():
        from google.colab import drive
        drive.mount('/content/drive')
    assert DRIVE_ASSET_ROOT.is_dir(), f'Missing persistent Drive asset directory: {DRIVE_ASSET_ROOT}'
    # Local SSD assets are initialized in the bootstrap cell below.
else:
    print('Safe default: RUN_STAGE=False; no setup, GPU job or training launched.')


Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Torch/CUDA: 2.11.0+cu130 13.0
GPU: NVIDIA L4
Free/total GiB: [21.84, 22.03]


## Reviewed GitHub source
A notebook already open in VS Code/Colab does not update when GitHub changes.
Reopen the reviewed notebook if it contains old cells. Existing modified source
is rejected; no local work is reset. Pin the full SHA for repeat runs/resume.


In [15]:
if RUN_STAGE:
    if not REPO.exists():
        stream(['git','clone','--single-branch','--branch',SOURCE_BRANCH,REPO_URL,REPO])
    assert git('remote','get-url','origin').removesuffix('.git') == REPO_URL.removesuffix('.git')
    assert git('branch','--show-current') == SOURCE_BRANCH
    assert not git('status','--porcelain','--untracked-files=no'), 'Modified source; preserve it and use a fresh checkout'
    stream(['git','-C',REPO,'fetch','origin',SOURCE_BRANCH])
    remote = git('rev-parse',f'origin/{SOURCE_BRANCH}')
    if EXPECTED_COMMIT:
        assert re.fullmatch(r'[0-9a-f]{40}',EXPECTED_COMMIT)
        assert remote == EXPECTED_COMMIT, 'Remote tip differs from the reviewed SHA'
    stream(['git','-C',REPO,'merge','--ff-only',f'origin/{SOURCE_BRANCH}'])
    SOURCE_REVISION = git('rev-parse','HEAD')
    assert SOURCE_REVISION == remote
    required = ['tools/train_rp010.py','utils/rp010.py','configs/rp010a.json',
                'configs/rp010b.json','docs/RP010_CONTROLLED_TRAINING_AUDIT.md','docs/RP010_RUNBOOK.md']
    assert all((REPO/p).is_file() for p in required), 'RP-010 files are not on GitHub yet: review, commit and push them first'
    print('Pinned source:',SOURCE_REVISION)


+ git -C /content/T2ICount-RP010 fetch origin rich-prompt-alignment
From https://github.com/bqa100507-spec/T2ICount-Implementation
 * branch            rich-prompt-alignment -> FETCH_HEAD
+ git -C /content/T2ICount-RP010 merge --ff-only origin/rich-prompt-alignment
Already up to date.
Pinned source: 9f673ce18fc3b2d90772f20275f2cd7827114806


## Dependencies, legacy asset ZIP bootstrap and persistent RP-010 outputs

Reuse the **same asset archive** as the working `train_colab.ipynb`:
`/content/drive/MyDrive/T2ICount-assets.zip`. Source pretrained weights are
extracted to `/content/T2ICount-assets/pretrained/` on the Colab SSD.
If local assets are complete, extraction is skipped.

Only the dedicated `checkpoints/rp010` and `outputs/rp010` paths under the local
asset root are symlinked to persistent Google Drive folders. No existing files
are moved/deleted and no pretrained checkpoint is redownloaded. Two arms use
exactly the same asset paths, initial checkpoint and source/runtime guards.

If installing packages requests a runtime restart, restart and rerun the whole
notebook before preparing the common initialization.


In [16]:
if RUN_STAGE:
    if INSTALL_DEPENDENCIES:
        stream([sys.executable, '-m', 'pip', 'install', '-r', REPO/'requirements-colab.txt'], cwd=REPO)
    taming = next(iter(REPO.glob('src/taming-transformers')), None)
    assert taming is not None, 'Pinned editable taming-transformers checkout missing'
    os.environ['PYTHONPATH'] = os.pathsep.join([str(REPO), str(taming), os.environ.get('PYTHONPATH', '')])
    for location in (REPO, taming):
        if str(location) not in sys.path:
            sys.path.insert(0, str(location))
    stream([sys.executable, '-c', 'import torch, torchvision, transformers, taming; '
            'from tools.train_rp010 import ControlledTrainer; print("Imports OK")'], cwd=REPO)

    # The old working train_colab.ipynb unpacked *all* pretrained/dataset assets
    # from this ZIP to the Colab SSD. RP-010 requires the same pretrained files.
    sd = ASSET_ROOT / 'pretrained/sd-v1-5/v1-5-pruned-emaonly.ckpt'
    clip = ASSET_ROOT / 'pretrained/clip-vit-large-patch14/config.json'
    if not (sd.is_file() and clip.is_file()):
        if not ASSET_ZIP.is_file():
            raise FileNotFoundError(
                f'Missing local SD/CLIP assets and legacy ZIP: {ASSET_ZIP}. '
                'Use the original T2ICount-assets.zip from the old training workflow; '
                'do not download arbitrary replacement checkpoints.')
        zip_size = ASSET_ZIP.stat().st_size
        assert zip_size > 0, 'The existing Drive asset archive is empty'
        if not LOCAL_ZIP.is_file() or LOCAL_ZIP.stat().st_size != zip_size:
            if LOCAL_ZIP.exists():
                LOCAL_ZIP.unlink()  # only incomplete local SSD copy; never touch Drive
            free = shutil.disk_usage('/content').free
            if free < zip_size + 5 * 1024**3:
                raise RuntimeError('Not enough Colab SSD space to copy the original asset ZIP')
            print('Copying existing T2ICount-assets.zip Drive -> Colab SSD...', flush=True)
            shutil.copyfile(ASSET_ZIP, LOCAL_ZIP)
        assert zipfile.is_zipfile(LOCAL_ZIP), f'Invalid local ZIP: {LOCAL_ZIP}'
        with zipfile.ZipFile(LOCAL_ZIP) as archive:
            members = archive.infolist()
            names = [item.filename.replace('\\', '/') for item in members]
            for name in names:
                parts = Path(name).parts
                if Path(name).is_absolute() or '..' in parts:
                    raise ValueError(f'Unsafe archive member: {name}')
            has_root = any(name.startswith('T2ICount-assets/') for name in names)
            total_bytes = sum(item.file_size for item in members)
        destination = Path('/content') if has_root else ASSET_ROOT
        destination.mkdir(parents=True, exist_ok=True)
        free = shutil.disk_usage('/content').free
        if free < total_bytes + 2 * 1024**3:
            raise RuntimeError('Not enough Colab SSD space to extract legacy asset ZIP')
        print('Extracting existing asset ZIP ->', destination, flush=True)
        stream(['unzip', '-q', '-o', LOCAL_ZIP, '-d', destination])
        if not (sd.is_file() and clip.is_file()):
            raise FileNotFoundError(
                f'ZIP extracted, but expected SD/CLIP assets are missing at {ASSET_ROOT}; '
                'check archive layout before changing anything.')
        LOCAL_ZIP.unlink()  # remove only SSD copy after successful extraction

    # Dataset can be extracted on SSD or exist already on persistent Drive.
    if DATA_ROOT is None:
        for candidate in (ASSET_ROOT / 'datasets/FSC147', DRIVE_ASSET_ROOT / 'datasets/FSC147'):
            if (candidate / 'FSC_147/Train_Test_Val_FSC_147.json').is_file():
                DATA_ROOT = candidate
                break
    assert DATA_ROOT is not None, 'FSC147 not found: check local ZIP extraction or set DATA_ROOT'
    DATA_ROOT = Path(DATA_ROOT)
    for path in (DATA_ROOT / 'FSC_147/Train_Test_Val_FSC_147.json', TRAIN_BANK, sd, clip):
        assert path.is_file(), f'Missing required asset: {path}'

    # The reviewed trainer configs still resolve pretrained from ASSET_ROOT.
    # Link only the RP-010 output subtrees to Drive, retaining all files there.
    def link_persistent_rp010(kind):
        local_link = ASSET_ROOT / kind / 'rp010'
        drive_target = DRIVE_ASSET_ROOT / kind / 'rp010'
        drive_target.mkdir(parents=True, exist_ok=True)
        local_link.parent.mkdir(parents=True, exist_ok=True)
        if local_link.is_symlink():
            if local_link.resolve() != drive_target.resolve():
                raise RuntimeError(f'Existing RP-010 symlink targets wrong folder: {local_link}')
        elif local_link.exists():
            raise RuntimeError(
                f'Existing non-symlink RP-010 directory at {local_link}. '
                'Preserve its contents and inspect manually; nothing has been deleted.')
        else:
            local_link.symlink_to(drive_target, target_is_directory=True)
        assert local_link.resolve() == drive_target.resolve()
        print(f'Persistent {kind}/rp010:', local_link, '->', drive_target)

    link_persistent_rp010('checkpoints')
    link_persistent_rp010('outputs')
    print('Pretrained SD:', sd)
    print('Pretrained CLIP:', clip.parent)
    print('Dataset:', DATA_ROOT)
    print('Training bank:', TRAIN_BANK)
    print('NOTE: RP-010 initialization, checkpoints and outputs persist on Drive.')


+ /usr/bin/python3 -m pip install -r /content/T2ICount-RP010/requirements-colab.txt
Obtaining taming-transformers from git+https://github.com/CompVis/taming-transformers.git@3ba01b241669f5ade541ce990f7650a3b8f65318#egg=taming-transformers (from -r /content/T2ICount-RP010/requirements-colab.txt (line 14))
  Skipping because already up-to-date.
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
Obtaining clip from git+https://github.com/openai/CLIP.git@d05afc436d78f1c48dc0dbf8e5980a9d471f35f6#egg=clip (from -r /content/T2ICount-RP010/requirements-cola

## Execute the selected stage
`prepare` creates one shared initialization and token audit. `smoke` runs one
compatible three-forward batch, real loss/backward/update, save/load, one-image
validation and a second resumed update. Run it once per arm; read both results.

`train` requires RUN_STAGE=True **and** ENABLE_FULL_TRAINING=True plus both
matching passed smoke reports. Checkpoints/logs are saved to Drive every completed
epoch. If interrupted before the first checkpoint, preserve that folder and use
a new PAIR_ID; otherwise set RESUME=True. A partial epoch is replayed.

`evaluate` accepts only completed runs and their validation-selected best weights;
it produces separate standard-test and FSC147-S class tables. `compare` verifies
paired control traces and joins per-image errors. It does not choose a winner.


In [17]:
if RUN_STAGE:
    assert SOURCE_REVISION and git('rev-parse','HEAD') == SOURCE_REVISION
    assert not git('status','--porcelain','--untracked-files=no')
    command = [sys.executable,'-u',REPO/'tools/train_rp010.py',STAGE,
               '--config',REPO/f'configs/rp010{ARM.lower()}.json',
               '--asset-root',ASSET_ROOT,'--seed',SEED,'--pair-id',PAIR_ID,
               '--data-dir',DATA_ROOT,'--prompt-bank',TRAIN_BANK]
    if STAGE == 'train':
        assert ENABLE_FULL_TRAINING, 'Review audit and both smokes, then explicitly enable full training'
        command += ['--allow-full-training']
        if RESUME:
            command += ['--resume']
    if STAGE == 'evaluate':
        assert S_BANK.is_file(), f'Missing existing evaluation bank: {S_BANK}'
        command += ['--s-metadata',REPO/'FSC-147-S.json','--s-bank',S_BANK]
    if STAGE == 'compare':
        command += ['--comparison-output',DRIVE_ASSET_ROOT/f'outputs/rp010/seed{SEED}/{PAIR_ID}/paired_analysis']
    stream(command,cwd=REPO)


+ /usr/bin/python3 -u /content/T2ICount-RP010/tools/train_rp010.py compare --config /content/T2ICount-RP010/configs/rp010a.json --asset-root /content/T2ICount-assets --seed 3407 --pair-id pair_v1 --data-dir /content/T2ICount-assets/datasets/FSC147 --prompt-bank /content/drive/MyDrive/T2ICount-assets/prompts/fsc147_train1000_seed3407_gemma4_v3.json --comparison-output /content/drive/MyDrive/T2ICount-assets/outputs/rp010/seed3407/pair_v1/paired_analysis


## Inspect artifacts before the next stage
Do not infer success from a notebook cell merely finishing. Missing/pending smoke,
failed validation, unexpected step count or changed source/assets blocks training.
Full-run completion is 10 epochs and 10,000 optimizer steps per arm, selected only
by validation MAE+RMSE at epochs7/8/9. Conditional bootstrap intervals describe
image uncertainty for this pair; replicate training seeds before stable claims.


In [18]:
if RUN_STAGE:
    run = DRIVE_ASSET_ROOT / f'outputs/rp010/seed{SEED}/{PAIR_ID}/RP-010{ARM}'
    if STAGE == 'prepare':
        report = DRIVE_ASSET_ROOT / f'checkpoints/rp010/init_seed{SEED}/initial_manifest.json'
        info = json.loads(report.read_text())
        print('Init SHA:', info['initialization_sha256'])
        print('GPU memory budget estimate (GiB):',
              info['memory']['weights_gradients_adam_estimate_bytes'] / 2**30)
        print('Token audit:', report.parent / 'prompt_tokens.csv')
    elif STAGE == 'smoke':
        report = run.with_name(run.name + '_smoke') / 'smoke_result.json'
        info = json.loads(report.read_text())
        print(json.dumps({k: v for k, v in info.items() if k not in ('guard', 'data')}, indent=2))
        assert info['status'] == 'passed', 'GPU smoke is pending/failed; do not launch full training'
    elif STAGE == 'train':
        info = json.loads((run / 'completed.json').read_text())
        assert info['epochs'] == 10 and info['optimizer_steps'] == 10000
        selected = run / info['selection']['checkpoint']
        assert selected.is_file() and (run / 'latest.tar').is_file()
        print('Completed; selected by validation:', info['selection'])
    elif STAGE == 'evaluate':
        print((run / 'evaluation_class/summary.json').read_text())
    else:
        print((run.parent / 'paired_analysis/summary.json').read_text())


{
  "fsc147_test_class.csv": {
    "a": {
      "n": 1190,
      "mae": 25.364190560729565,
      "rmse": 122.19738851141194,
      "bias": 10.52995026301937,
      "overcount": 1052,
      "undercount": 138,
      "ties": 0
    },
    "b": {
      "n": 1190,
      "mae": 22.304271636890764,
      "rmse": 118.76643689975273,
      "bias": 8.198283176462189,
      "overcount": 1046,
      "undercount": 144,
      "ties": 0
    },
    "b_minus_a": {
      "mae": -3.0599189238388007,
      "rmse": -3.4309516116592107,
      "bias": -2.3316670865571805
    },
    "mae_delta_image_bootstrap95": [
      -4.339421670835058,
      -1.7975059288114064
    ],
    "uncertainty": "conditional on this trained pair; not training-seed uncertainty"
  },
  "fsc147s_class.csv": {
    "a": {
      "n": 230,
      "mae": 28.19786112256672,
      "rmse": 53.21130634909327,
      "bias": 27.788950188781904,
      "overcount": 223,
      "undercount": 7,
      "ties": 0
    },
    "b": {
      "n": 230,
    